# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [ ]:
def transposee(A):
    lignes = len(A)
    colonnes = len(A[0])
    return [[A[i][j] for i in range(lignes)] for j in range(colonnes)]
 
print(transposee([[1, 2, 3], [4, 5, 6]]))

## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [ ]:
def produit(A, B):
    n, p, m = len(A), len(B), len(B[0])
    C = [[0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            for k in range(p):
                C[i][j] = C[i][j] + A[i][k] * B[k][j]
    return C


## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [ ]:
def aleatoire(n):
    return [[random.random() for _ in range(n)] for _ in range(n)]
 
def chrono(f, *args):
    debut = time.perf_counter()
    f(*args)
    return time.perf_counter() - debut
 
precedent = None
for n in [40, 80, 120]:
    M, N = aleatoire(n), aleatoire(n)
    t = chrono(produit, M, N)
    if precedent:
        print(n, t, "rapport avec la taille precedente :", t / precedent)
    else:
        print(n, t)
    precedent = t


## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [ ]:
n = 120
M, N = aleatoire(n), aleatoire(n)
tp = chrono(produit, M, N)
Mn, Nn = np.array(M), np.array(N)
tn = chrono(lambda: Mn @ Nn)
print("python", tp, "numpy", tn, "rapport", tp / tn)


## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [ ]:
def resoudre(A, b):
    n = len(A)
    A = [ligne[:] for ligne in A]     
    b = b[:]
    for k in range(n):
        p = k
        for i in range(k + 1, n):
            if abs(A[i][k]) > abs(A[p][k]):
                p = i
        A[k], A[p] = A[p], A[k]
        b[k], b[p] = b[p], b[k]
        tol = 0 if isinstance(A[k][k], Fraction) else 1e-12
        if abs(A[k][k]) <= tol:
            raise ValueError("systeme singulier")
        for i in range(k + 1, n):
            facteur = A[i][k] / A[k][k]
            for j in range(k, n):
                A[i][j] = A[i][j] - facteur * A[k][j]
            b[i] = b[i] - facteur * b[k]
    x = [0] * n
    for i in range(n - 1, -1, -1):
        s = b[i]
        for j in range(i + 1, n):
            s = s - A[i][j] * x[j]
        x[i] = s / A[i][i]
    return x


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [ ]:
n = 50
A = aleatoire(n)
b = [random.random() for _ in range(n)]
x_moi = resoudre(A, b)
x_np = np.linalg.solve(np.array(A), np.array(b))
print("ecart max :", max(abs(x_moi[i] - x_np[i]) for i in range(n)))


## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [ ]:
# Le bloc de l'enonce n'est pas fourni. Exemple typique : Hilbert 12 en
# flottants. numpy se trompe beaucoup, la methode exacte (Fraction) donne la
# vraie solution. Le conditionnement mesure l'amplification des erreurs :
H = np.array([[1 / (i + j + 1) for j in range(12)] for i in range(12)])
print("conditionnement :", np.linalg.cond(H))   # environ 1e16
# cond ~ 1e16 : les erreurs d'arrondi (1e-16) peuvent etre multipliees par
# 1e16, donc plus aucun chiffre n'est fiable.
print(np.linalg.solve(H, H.sum(axis=1)))  


## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [ ]:
def rotation(angle_deg):
    a = math.radians(angle_deg)
    return [[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]
 
R = produit(rotation(60), rotation(30))
R90 = rotation(90)
print(R)
print(R90)
print(np.allclose(R, R90))


## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [ ]:
def determinant(A):
    n = len(A)
    A = [ligne[:] for ligne in A]
    det = 1
    for k in range(n):
        p = k
        for i in range(k + 1, n):
            if abs(A[i][k]) > abs(A[p][k]):
                p = i
        if A[p][k] == 0:
            return 0
        if p != k:
            A[k], A[p] = A[p], A[k]
            det = -det                 
        det = det * A[k][k]            
        for i in range(k + 1, n):
            facteur = A[i][k] / A[k][k]
            for j in range(k, n):
                A[i][j] = A[i][j] - facteur * A[k][j]
    return det

## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [ ]:
# votre réponse ici
# Le premier essai echoue parce que le test "pivot proche de 0" (< 1e-12)
# declare "singulier" des pivots tres petits mais NON nuls : pour Hilbert,
# ils deviennent minuscules. Avec des Fraction il n'y a aucun arrondi, donc
# on teste seulement "pivot == 0" (c'est ce que fait resoudre ci-dessus).
n = 12
H = [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]
b = [sum(ligne) for ligne in H]
print(resoudre(H, b))

## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [ ]:
def translation(tx, ty):
    return [[1, 0, tx], [0, 1, ty], [0, 0, 1]]
 
def rotation_h(angle):
    a = math.radians(angle)
    return [[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]]
